In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import log_loss
from sklearn.preprocessing import LabelEncoder, StandardScaler

# 1. Carregamento dos dados
train = pd.read_csv("/kaggle/input/competitions/campeonato-inteli-modulo-3-2026/train.csv")
test = pd.read_csv("/kaggle/input/competitions/campeonato-inteli-modulo-3-2026/test.csv")

# 2. Remoção Estrita de Vazamento (Leakage)
train_clean = train.drop(columns=['odd_casa', 'odd_empate', 'odd_visitante'])

le = LabelEncoder()
y_train = le.fit_transform(train_clean['Res'])

# 3. Engenharia de Features Avançada (Criando Interações)
def prepare_features(df):
    df_feat = df.copy()
    
    # Deltas Básicos
    df_feat['elo_diff'] = df_feat['elo_home'] - df_feat['elo_away']
    df_feat['form_pts_diff'] = df_feat['form_pts_home'] - df_feat['form_pts_away']
    df_feat['season_ppg_diff'] = df_feat['season_ppg_home'] - df_feat['season_ppg_away']
    df_feat['form_gf_diff'] = df_feat['form_gf_home'] - df_feat['form_gf_away']
    df_feat['form_ga_diff'] = df_feat['form_ga_home'] - df_feat['form_ga_away']
    
    # TÉCNICA 1: INTERAÇÃO (Momentum + Qualidade)
    df_feat['elo_x_form'] = df_feat['elo_diff'] * df_feat['form_pts_diff']
    
    features = [
        'elo_diff', 'form_pts_diff', 'season_ppg_diff', 
        'form_gf_diff', 'form_ga_diff', 'elo_x_form',
        'elo_home', 'elo_away', 'rest_days_home', 'rest_days_away'
    ]
    return df_feat[features]

X_train_full = prepare_features(train_clean).fillna(0)
X_test_final = prepare_features(test).fillna(0)

# Escalonamento Rigoroso
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_full)
X_test_scaled = scaler.transform(X_test_final)

# TÉCNICA 2: PESOS TEMPORAIS (Decay Weight)
# Dá mais importância (peso maior) para os anos recentes (2020, 2021) do que para 2013
ano_maximo = train_clean['Season'].max()
pesos_temporais = np.exp((train_clean['Season'] - ano_maximo) / 4)


# TREINAMENTO DE 3 MODELOS ROBUSTOS PARA O BLEND FINAL
print("Treinando Modelos com Calibração Isotônica e Pesos Temporais...")

# Modelo 1: Regressão Logística (Espinha dorsal do LogLoss)
model_lr = LogisticRegression(C=0.01, max_iter=2000, random_state=42)
cal_lr = CalibratedClassifierCV(model_lr, method='isotonic', cv=5)
cal_lr.fit(X_train_scaled, y_train, sample_weight=pesos_temporais)
preds_lr = cal_lr.predict_proba(X_test_scaled)

# Modelo 2: HistGradientBoosting (Captura nuances profundas)
model_hgb = HistGradientBoostingClassifier(learning_rate=0.01, max_depth=4, l2_regularization=3.0, max_iter=300, random_state=42)
cal_hgb = CalibratedClassifierCV(model_hgb, method='isotonic', cv=5)
cal_hgb.fit(X_train_scaled, y_train, sample_weight=pesos_temporais)
preds_hgb = cal_hgb.predict_proba(X_test_scaled)

# Modelo 3: Random Forest (Árvores independentes para evitar overfitting)
model_rf = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=15, random_state=42)
cal_rf = CalibratedClassifierCV(model_rf, method='isotonic', cv=5)
cal_rf.fit(X_train_scaled, y_train, sample_weight=pesos_temporais)
preds_rf = cal_rf.predict_proba(X_test_scaled)


# TÉCNICA 3: ENSEMBLE (MISTURA INTELIGENTE)

# Misturamos as previsões dando mais peso para o modelo linear (mais seguro)
# e pesos menores para as árvores (que acertam as zebras não lineares)
preds_finais = (preds_lr * 0.55) + (preds_hgb * 0.25) + (preds_rf * 0.20)

# Montagem e Salvação do CSV
submission = pd.DataFrame({
    'Id': test['Id'],
    'A': preds_finais[:, 0], # A = Visitante
    'D': preds_finais[:, 1], # D = Empate
    'H': preds_finais[:, 2]  # H = Mandante
})

submission.to_csv("ali-submissao3.csv", index=False)
print("Arquivo 'ali-submissao3.csv' gerado! Modelo de alta capacidade preditiva concluído.")

Treinando Modelos com Calibração Isotônica e Pesos Temporais...
Arquivo 'ali-submissao3.csv' gerado! Modelo de alta capacidade preditiva concluído.
